<a href="https://colab.research.google.com/github/maucikamau/GiftHub/blob/main/asd2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Naslov rada


## Imports

In [2]:
import glob
import warnings
import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from numpy import interp
from itertools import cycle

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, roc_curve, auc, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

import matplotlib.pyplot as plt
import seaborn as sns

# Opcionalno: yellowbrick za learning curve
try:
    from yellowbrick.model_selection import LearningCurve
    YELLOWBRICK_AVAILABLE = True
except ImportError:
    YELLOWBRICK_AVAILABLE = False
    print("Yellowbrick nije instaliran. Learning curve će koristiti sklearn.")

warnings.filterwarnings('ignore')

## Pomoćne funkcije

In [3]:
def timeseries_to_connectivity(csv_path):
    """Pretvori vremenske serije u Spearman connectivity matricu."""
    df = pd.read_csv(csv_path, header=None)
    data = df.values
    corr_matrix, _ = spearmanr(data)
    corr_matrix = np.nan_to_num(corr_matrix)
    np.fill_diagonal(corr_matrix, 0)
    return corr_matrix


def flatten_upper_triangle(matrix):
    """Ekstrahiraj gornji trokut matrice (bez dijagonale)."""
    arr = np.array(matrix)
    n = arr.shape[0]
    triu_indices = np.triu_indices(n, k=1)
    return arr[triu_indices]

In [25]:
import numpy as np
from scipy.stats import spearmanr

def get_features_and_labels(abide_dataset, derivative_name):
    """
    Automatizira izračun Spearmanovih matrica i pripremu labela.
    """
    features_list = []
    labels_list = []

    # Pristupamo dinamički (npr. abide_dataset.rois_cc200)
    timeseries_data = getattr(abide_dataset, derivative_name)
    pheno_data = abide_dataset.phenotypic

    print(f"Obrađujem: {derivative_name}...")

    for i, (ts, label) in enumerate(zip(timeseries_data, pheno_data['DX_GROUP'])):
        # Ako je ts putanja do datoteke, učitaj je
        if isinstance(ts, str):
            ts = np.loadtxt(ts)

        # Izračun Spearmanove korelacije
        # spearmanr računa korelaciju po stupcima (regijama)
        corr_matrix, _ = spearmanr(ts)

        # Čišćenje matrice
        corr_matrix = np.nan_to_num(corr_matrix)
        np.fill_diagonal(corr_matrix, 0)

        # Vektorizacija (uzimanje gornjeg trokuta)
        # Pretpostavljam da imaš funkciju flatten_upper_triangle definiranu
        flat_features = flatten_upper_triangle(corr_matrix)

        features_list.append(flat_features)
        labels_list.append(1 if label == 1 else 0) # 1=ASD, 0=TD

        if (i+1) % 200 == 0:
            print(f"  Obrađeno {i+1} subjekata...")

    return np.array(features_list), np.array(labels_list)

## Vizualizacijske funkcije

In [4]:
def plot_confusion_matrix_detailed(y_true, y_pred, title, save_path=None):
    """
    Nacrtaj detaljni confusion matrix s postocima.
    """
    list_names = ['TD', 'ASD']
    cm = confusion_matrix(y_true, y_pred)
    cm_sum = np.sum(cm, axis=1, keepdims=True)
    cm_perc = cm / cm_sum.astype(float) * 100

    annot = np.empty_like(cm).astype(str)
    nrows, ncols = cm.shape

    for i in range(nrows):
        for j in range(ncols):
            c = cm[i, j]
            p = cm_perc[i, j]
            if i == j:
                s = cm_sum[i]
                annot[i, j] = '%.1f%%\n%d/%d' % (p, c, s)
            elif c == 0:
                annot[i, j] = ''
            else:
                annot[i, j] = '%.1f%%\n%d' % (p, c)

    cm_df = pd.DataFrame(cm)
    cm_df.index.name = 'Actual'
    cm_df.columns.name = 'Predicted'

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=annot, fmt='', cmap='rocket_r',
                xticklabels=list_names, yticklabels=list_names)
    plt.title(title)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    return cm

In [5]:
def plot_roc_curve_multiclass(y_true, y_pred, y_prob=None, title='ROC Curve', save_path=None):
    """
    Nacrtaj ROC krivulju s macro prosjekom za binarnu klasifikaciju.
    """
    list_names = ['TD', 'ASD']
    n_classes = 2

    # Ako nemamo probabilitete, koristimo predikcije
    if y_prob is None:
        y_prob = y_pred

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    # One-hot encoding za ROC
    y_true_onehot = np.array(pd.get_dummies(y_true))
    y_pred_onehot = np.array(pd.get_dummies(y_pred))

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_onehot[:, i], y_pred_onehot[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Macro average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes

    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    # Plot
    lw = 2
    plt.figure(figsize=(8, 6))

    plt.plot(fpr["macro"], tpr["macro"],
             label='Macro-average ROC (AUC = {0:0.2f})'.format(roc_auc["macro"]),
             color='slategray', linestyle=':', linewidth=4)

    colors = cycle(['purple', 'lightseagreen'])
    for i, color, name in zip(range(n_classes), colors, list_names):
        plt.plot(fpr[i], tpr[i], color=color, lw=lw,
                 label='ROC curve - {0} (AUC = {1:0.2f})'.format(name, roc_auc[i]))

    plt.plot([0, 1], [0, 1], 'k--', color='#cb416b', lw=lw)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.annotate(' Random Guess', (.5, .48), color='#cb416b')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc='lower right')

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    return roc_auc

In [6]:
def plot_learning_curve_sklearn(estimator, X, y, title='Learning Curve',
                                 cv=10, n_jobs=-1, save_path=None):
    """
    Nacrtaj learning curve koristeći sklearn.
    """
    train_sizes = np.linspace(0.1, 1.0, 10)

    train_sizes_abs, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs,
        train_sizes=train_sizes, scoring='accuracy'
    )

    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)

    plt.figure(figsize=(10, 6))

    plt.fill_between(train_sizes_abs, train_mean - train_std,
                     train_mean + train_std, alpha=0.1, color='steelblue')
    plt.fill_between(train_sizes_abs, test_mean - test_std,
                     test_mean + test_std, alpha=0.1, color='coral')

    plt.plot(train_sizes_abs, train_mean, 'o-', color='steelblue',
             label='Training score')
    plt.plot(train_sizes_abs, test_mean, 'o-', color='coral',
             label='Cross-validation score')

    plt.xlabel('Training Instances')
    plt.ylabel('Score')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [7]:
def plot_learning_curve_yellowbrick(estimator, X, y, title='Learning Curve',
                                     cv=10, save_path=None):
    """
    Nacrtaj learning curve koristeći yellowbrick.
    """
    if not YELLOWBRICK_AVAILABLE:
        print("Yellowbrick nije dostupan, koristim sklearn verziju.")
        plot_learning_curve_sklearn(estimator, X, y, title, cv, save_path=save_path)
        return

    sizes = np.linspace(0.1, 1, 10)
    visualizer = LearningCurve(
        estimator, cv=cv, scoring='accuracy', train_sizes=sizes, n_jobs=-1
    )
    visualizer.fit(X, y)

    if save_path:
        visualizer.show(outpath=save_path)
    else:
        visualizer.show()

# Funkcija treniranje i evaulacije

In [8]:
def train_and_evaluate(X_train, y_train, X_test, y_test, dataset_name):
    """
    Treniraj modele s GridSearchCV i evaluiraj na test setu.
    Vraća DataFrame s rezultatima i najbolje modele.
    """

    models = {
        'Logistic Regression': {
            'model': LogisticRegression(random_state=42, max_iter=1000),
            'params': {'C': [0.01, 0.1, 1, 10]}
        },
        'Random Forest': {
            'model': RandomForestClassifier(random_state=42),
            'params': {'n_estimators': [100, 200], 'max_depth': [10, 20, None]}
        },
        'Naive Bayes': {
            'model': GaussianNB(),
            'params': {'var_smoothing': [1e-9, 1e-7, 1e-5]}
        },
        'MLP': {
            'model': MLPClassifier(random_state=42, max_iter=1000),
            'params': {'hidden_layer_sizes': [(100,), (100, 50)], 'alpha': [0.0001, 0.001]}
        }
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []
    best_models = {}
    predictions = {}

    for name, config in models.items():
        print(f"\nTreniram {name}...")

        grid = GridSearchCV(config['model'], config['params'],
                           cv=cv, scoring='roc_auc', n_jobs=-1)
        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_
        best_models[name] = best_model

        y_pred = best_model.predict(X_test)

        # Probabiliteti za ROC
        if hasattr(best_model, 'predict_proba'):
            y_prob = best_model.predict_proba(X_test)[:, 1]
        else:
            y_prob = y_pred

        predictions[name] = {
            'y_pred': y_pred,
            'y_prob': y_prob
        }

        results.append({
            'Model': name,
            'CV AUC': grid.best_score_,
            'Test AUC': roc_auc_score(y_test, y_prob),
            'Accuracy': accuracy_score(y_test, y_pred),
            'F1': f1_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred),
            'Recall': recall_score(y_test, y_pred),
            'Best Params': str(grid.best_params_)
        })

        print(f"  CV AUC: {grid.best_score_:.3f}, Test AUC: {results[-1]['Test AUC']:.3f}")

    df_results = pd.DataFrame(results)

    print(f"\n{'='*60}")
    print(f"REZULTATI - {dataset_name}")
    print('='*60)
    print(df_results[['Model', 'CV AUC', 'Test AUC', 'Accuracy', 'F1']].to_string(index=False))

    return df_results, best_models, predictions

# Treniranje na ABIDE skupu
Korištenje 3 načina pretprocesiranja (CPAC, CCS, NIAK)

### 1.1 Učitavanje podataka

In [11]:
!pip install nilearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 83.6 MB/s eta 0:00:00


In [24]:
from nilearn.datasets import fetch_abide_pcp
from google.colab import drive
import os

drive.mount('/content/drive')

# izmijenjeno i ponovljeno za gsr=True

dir_path1 = '/content/drive/MyDrive/ABIDE_gsrFalse'
if not os.path.exists(dir_path1):
    os.makedirs(dir_path1)


abide1 = fetch_abide_pcp(
    data_dir=dir_path1,
    derivatives=['rois_cc200'],
    pipeline='cpac',
    band_pass_filtering=True,
    global_signal_regression=False,
    quality_checked=True
)
print(f"[CPAC] Broj subjekata: {len(abide1.rois_cc200)}")
print(f"[CPAC] ASD: {sum(abide1.phenotypic['DX_GROUP']==1)}, TD: {sum(abide1.phenotypic['DX_GROUP']==2)}")

abide2 = fetch_abide_pcp(
    data_dir=dir_path1,
    derivatives=['rois_cc200'],
    pipeline='ccs',
    band_pass_filtering=True,
    global_signal_regression=False,
    quality_checked=True
)
print(f"[CCS] Broj subjekata: {len(abide2.rois_cc200)}")
print(f"[CCS] ASD: {sum(abide2.phenotypic['DX_GROUP']==1)}, TD: {sum(abide2.phenotypic['DX_GROUP']==2)}")

abide3 = fetch_abide_pcp(
    data_dir=dir_path1,
    derivatives=['rois_cc200'],
    pipeline='niak',
    band_pass_filtering=True,
    global_signal_regression=False,
    quality_checked=True
)
print(f"[NIAK] Broj subjekata: {len(abide3.rois_cc200)}")
print(f"[NIAK] ASD: {sum(abide3.phenotypic['DX_GROUP']==1)}, TD: {sum(abide3.phenotypic['DX_GROUP']==2)}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[fetch_abide_pcp] Dataset found in /content/drive/MyDrive/ABIDE_gsrFalse/ABIDE_pcp

[CPAC] Broj subjekata: 871
[CPAC] ASD: 403, TD: 468


[fetch_abide_pcp] Dataset found in /content/drive/MyDrive/ABIDE_gsrFalse/ABIDE_pcp

[CCS] Broj subjekata: 871
[CCS] ASD: 403, TD: 468


[fetch_abide_pcp] Dataset found in /content/drive/MyDrive/ABIDE_gsrFalse/ABIDE_pcp

[NIAK] Broj subjekata: 871
[NIAK] ASD: 403, TD: 468


In [26]:
# Rječnik u kojem čuvaš svoje učitane datasetove
datasets_dict = {
    "CPAC": abide1,
    "CCS": abide2,
    "NIAK": abide3
}

# Rječnik u koji će se spremiti rezultati (X i y)
results = {}

for name, data in datasets_dict.items():
    print(f"\n--- Pokrećem obradu za pipeline: {name} ---")

    # Pozivamo našu funkciju
    X, y = get_features_and_labels(data, 'rois_cc200')

    # Spremamo rezultate da ih kasnije možemo pozvati
    results[name] = {'X': X, 'y': y}

    print(f"Završeno za {name}. X shape: {X.shape}")


--- Pokrećem obradu za pipeline: CPAC ---
Obrađujem: rois_cc200...
  Obrađeno 200 subjekata...
  Obrađeno 400 subjekata...
  Obrađeno 600 subjekata...
  Obrađeno 800 subjekata...
Završeno za CPAC. X shape: (871, 19900)

--- Pokrećem obradu za pipeline: CCS ---
Obrađujem: rois_cc200...
  Obrađeno 200 subjekata...
  Obrađeno 400 subjekata...
  Obrađeno 600 subjekata...
  Obrađeno 800 subjekata...
Završeno za CCS. X shape: (871, 19900)

--- Pokrećem obradu za pipeline: NIAK ---
Obrađujem: rois_cc200...
  Obrađeno 200 subjekata...
  Obrađeno 400 subjekata...
  Obrađeno 600 subjekata...
  Obrađeno 800 subjekata...
Završeno za NIAK. X shape: (871, 19900)


## 1.2 Računanje matrica povezanosti

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Rječnik u koji ćemo spremiti finalne, skalirane podatke
processed_data = {}

print("Započinjem podjelu i skaliranje podataka...")

for name, data in results.items():
    X = data['X']
    y = data['y']

    # 1. Podjela na trening i test skup
    # Koristimo stratify=y kako bismo zadržali isti omjer ASD/TD u oba skupa
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=42
    )

    # 2. Skaliranje (Standardizacija)
    # VAŽNO: scaler učimo (fit) samo na trening skupu kako bismo izbjegli Data Leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 3. Spremanje u rječnik pod nazivom pipelinea
    processed_data[name] = {
        'X_train': X_train_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train,
        'y_test': y_test,
        'scaler': scaler # spremamo i scaler ako nam zatreba za ABIDE II kasnije
    }

    print(f"[{name}] Gotovo. Train: {X_train_scaled.shape[0]}, Test: {X_test_scaled.shape[0]}")

print("\nSvi podaci su spremni za SVM model!")

Započinjem podjelu i skaliranje podataka...
[CPAC] Gotovo. Train: 653, Test: 218
[CCS] Gotovo. Train: 653, Test: 218
[NIAK] Gotovo. Train: 653, Test: 218

Svi podaci su spremni za SVM model!


In [ ]:
# Rječnici za spremanje svih rezultata
all_results = {}
all_best_models = {}
all_predictions = {}

print("Započinjem treniranje i evaluaciju za sve pipelineove...\n")

for name, data in processed_data.items():
    print(f"--- Evaluacija pipelinea: {name} ---")

    # Pozivamo tvoju funkciju train_and_evaluate s podacima iz rječnika
    results, best_models, predictions = train_and_evaluate(
        data['X_train'],
        data['y_train'],
        data['X_test'],
        data['y_test'],
        f"ABIDE (CC200) - {name}" # Dinamički naziv za grafove/ispis
    )

    # Spremanje rezultata za kasniju usporedbu
    all_results[name] = results
    all_best_models[name] = best_models
    all_predictions[name] = predictions

    print(f"Završeno za {name}.\n")

print("Sva testiranja su uspješno izvršena!")